In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.dummy import DummyRegressor
import xgboost as xgb
import lightgbm as lgb

In [ ]:
# Load data
df = pd.read_csv('housing_data_after_feature_selection.csv').drop(
    columns=['url','image_url','address','nearby_cities']
)

# Remove Unnamed: 0 if it exists
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

# Identify feature types
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f"Found categorical columns: {categorical_cols}")

numerical_cols = df.columns

Found categorical columns: ['property_type', 'region', 'zip_code', 'county']


In [ ]:
print(f"Dataset shape: {df.shape}")
print(f"Categorical columns: {categorical_cols}")
print(f"Numerical columns: {numerical_cols}")

Dataset shape: (7792, 17)
Categorical columns: ['property_type', 'region', 'zip_code', 'county']
Numerical columns: Index(['beds', 'baths', 'sqft', 'property_type', 'region',
       'parking_total_spaces', 'walk_score', 'middle_school_distance',
       'wind_risk', 'zip_code', 'mobility_score', 'parking_quality_score',
       'has_garage', 'county', 'price_volatility', 'price_reduction_total',
       'price'],
      dtype='object')


In [ ]:
X = df.drop('price', axis=1)
y = df['price']

In [ ]:
models = {
    'Dummy (Mean)': DummyRegressor(strategy='mean'),
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(),
    'Lasso': Lasso(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Extra Trees': ExtraTreesRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'XGBoost': xgb.XGBRegressor(random_state=42, verbosity=0),
    'LightGBM': lgb.LGBMRegressor(random_state=42, verbosity=-1),
    'KNN': KNeighborsRegressor(),
    'SVR': SVR()
}

In [ ]:
from sklearn.pipeline import Pipeline

def test_models(X_data, y_data, encoding_name, preprocessor=None):
    print(f"\n{encoding_name.upper()} ENCODING RESULTS:")
    # Feature shape print will be done inside the loop if preprocessor is used
    print("-" * 50)

    results = {}
    for name, model in models.items():
        try:
            if preprocessor:
                # Create a pipeline that first preprocesses, then models
                pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                           ('regressor', model)])
                scores = cross_val_score(pipeline, X_data, y_data, cv=5, scoring='r2')
                if name == list(models.keys())[0]: # Print shape only for the first model if preprocessor is used
                     print(f"Feature shape after preprocessing: {pipeline['preprocessor'].transform(X_data).shape}")
            else:
                # If no preprocessor, use the data directly
                scores = cross_val_score(model, X_data, y_data, cv=5, scoring='r2')
                if name == list(models.keys())[0]: # Print shape only for the first model if no preprocessor
                     print(f"Feature shape: {X_data.shape}")


            mean_score = scores.mean()
            std_score = scores.std()
            results[name] = mean_score
            print(f"{name:20}: R² = {mean_score:.4f} (+/- {std_score*2:.4f})")
        except Exception as e:
            print(f"{name:20}: Error - {str(e)}")
            results[name] = np.nan

    return results

In [ ]:
print("PART 1: ORDINAL ENCODING")

X_ordinal = X.copy()
for col in categorical_cols:
    if col in X_ordinal.columns:
        oe = OrdinalEncoder()
        X_ordinal[col] = oe.fit_transform(X_ordinal[[col]])

ordinal_results = test_models(X_ordinal, y, "ordinal")

PART 1: ORDINAL ENCODING

ORDINAL ENCODING RESULTS:
Feature shape: (7792, 16)
--------------------------------------------------
Dummy (Mean)        : R² = -0.1882 (+/- 0.6240)
Linear Regression   : R² = 0.1380 (+/- 1.9034)
Ridge               : R² = 0.1380 (+/- 1.9033)
Lasso               : R² = 0.1380 (+/- 1.9034)
Decision Tree       : R² = -0.0185 (+/- 1.6796)
Random Forest       : R² = 0.4990 (+/- 0.6687)
Extra Trees         : R² = 0.5827 (+/- 0.3872)
Gradient Boosting   : R² = 0.4284 (+/- 1.0168)
XGBoost             : R² = 0.5534 (+/- 0.2901)
LightGBM            : R² = 0.5104 (+/- 0.3868)
KNN                 : R² = 0.3128 (+/- 1.0836)
SVR                 : R² = -0.0933 (+/- 0.1486)


In [ ]:
print("\nPART 2: ONE-HOT ENCODING")

# Apply one-hot encoding
X_onehot = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

onehot_results = test_models(X_onehot, y, "one-hot")


PART 2: ONE-HOT ENCODING

ONE-HOT ENCODING RESULTS:
Feature shape: (7792, 919)
--------------------------------------------------
Dummy (Mean)        : R² = -0.1882 (+/- 0.6240)
Linear Regression   : R² = -0.8732 (+/- 3.6674)


/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=1.76056e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=2.13787e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=2.5213e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=1.36776e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=1.24346e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)


Ridge               : R² = -0.3494 (+/- 2.3911)


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.525e+15, tolerance: 2.601e+12
  model = cd_fast.enet_coordinate_descent(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.224e+15, tolerance: 2.449e+12
  model = cd_fast.enet_coordinate_descent(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.883e+15, tolerance: 3.752e

Lasso               : R² = -1.4189 (+/- 5.6770)
Decision Tree       : R² = 0.2265 (+/- 0.6305)
Random Forest       : R² = 0.5580 (+/- 0.5020)
Extra Trees         : R² = 0.6116 (+/- 0.1135)
Gradient Boosting   : R² = 0.3735 (+/- 0.8656)
XGBoost             : R² = 0.4802 (+/- 0.4364)
LightGBM            : R² = 0.4668 (+/- 0.6347)
KNN                 : R² = 0.2937 (+/- 1.1354)
SVR                 : R² = -0.0933 (+/- 0.1486)


In [ ]:
print("\nPART 3: MIXED ENCODING")

# Define mixed encoding strategy
# Ordinal for features with natural ordering
ordinal_features = {}  # Add any ordinal features here if you identify them

# One-hot for nominal features
onehot_features = [col for col in categorical_cols if col not in ordinal_features]
ordinal_features_mixed = ['region', 'zip_code', 'county']  # Just a list
onehot_features_mixed = ['property_type']



PART 3: MIXED ENCODING


In [ ]:
# PART 3: MIXED ENCODING (CORRECT)
numerical_cols_mixed = [col for col in X.columns if col not in categorical_cols + ['price']]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numerical_cols_mixed),
        ('onehot', OneHotEncoder(drop='first', sparse_output=False), onehot_features_mixed),
        ('ordinal', OrdinalEncoder(), ordinal_features_mixed)
    ],
    remainder='drop'
)

X_mixed = preprocessor.fit_transform(X)
print(f"Mixed encoding shape: {X_mixed.shape}")

mixed_results = test_models(X_mixed, y, "mixed")

Mixed encoding shape: (7792, 17)

MIXED ENCODING RESULTS:
Feature shape: (7792, 17)
--------------------------------------------------
Dummy (Mean)        : R² = -0.1882 (+/- 0.6240)
Linear Regression   : R² = 0.1344 (+/- 1.9163)
Ridge               : R² = 0.1344 (+/- 1.9163)
Lasso               : R² = 0.1344 (+/- 1.9163)
Decision Tree       : R² = -0.8029 (+/- 3.3904)
Random Forest       : R² = 0.4656 (+/- 0.6414)
Extra Trees         : R² = 0.5565 (+/- 0.3784)
Gradient Boosting   : R² = 0.4173 (+/- 0.9364)
XGBoost             : R² = 0.5457 (+/- 0.3436)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


LightGBM            : R² = 0.4881 (+/- 0.4692)
KNN                 : R² = 0.3128 (+/- 1.0836)
SVR                 : R² = -0.0933 (+/- 0.1486)


In [ ]:
print("\nCOMPARISON SUMMARY:")
print("-" * 70)

# Create comparison DataFrame
comparison_data = []
for model_name in models.keys():
    comparison_data.append({
        'Model': model_name,
        'Ordinal': ordinal_results.get(model_name, np.nan),
        'One-Hot': onehot_results.get(model_name, np.nan),
        'Mixed': mixed_results.get(model_name, np.nan)
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.round(4)

# Sort by best overall performance
comparison_df['Best_Score'] = comparison_df[['Ordinal', 'One-Hot', 'Mixed']].max(axis=1)
comparison_df = comparison_df.sort_values('Best_Score', ascending=False)

print(comparison_df[['Model', 'Ordinal', 'One-Hot', 'Mixed']].to_string(index=False))



COMPARISON SUMMARY:
----------------------------------------------------------------------
            Model  Ordinal  One-Hot   Mixed
      Extra Trees   0.5827   0.6116  0.5565
    Random Forest   0.4990   0.5580  0.4656
          XGBoost   0.5534   0.4802  0.5457
         LightGBM   0.5104   0.4668  0.4881
Gradient Boosting   0.4284   0.3735  0.4173
              KNN   0.3128   0.2937  0.3128
    Decision Tree  -0.0185   0.2265 -0.8029
Linear Regression   0.1380  -0.8732  0.1344
            Ridge   0.1380  -0.3494  0.1344
            Lasso   0.1380  -1.4189  0.1344
              SVR  -0.0933  -0.0933 -0.0933
     Dummy (Mean)  -0.1882  -0.1882 -0.1882


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import GridSearchCV, cross_val_score, validation_curve
from sklearn.metrics import mean_squared_error, r2_score
import time

# Prepare your best dataset (one-hot encoded)
X_onehot = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

# Baseline performance
baseline_et = ExtraTreesRegressor(n_estimators=100, random_state=42)
baseline_scores = cross_val_score(baseline_et, X_onehot, y, cv=5, scoring='r2')

# Parameter grid for GridSearchCV
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2']
}

# GridSearchCV
grid_search = GridSearchCV(
    estimator=ExtraTreesRegressor(random_state=42, n_jobs=-1),
    param_grid=param_grid,
    scoring='r2',
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_onehot, y)

# Best model evaluation
best_et = grid_search.best_estimator_
best_scores = cross_val_score(best_et, X_onehot, y, cv=5, scoring='r2')
improvement = best_scores.mean() - baseline_scores.mean()


Fitting 5 folds for each of 48 candidates, totalling 240 fits


In [ ]:
# Individual parameter analysis
n_est_range = [50, 100, 200, 300, 500]
n_est_scores = []

for n_est in n_est_range:
    et = ExtraTreesRegressor(n_estimators=n_est, random_state=42, n_jobs=-1)
    scores = cross_val_score(et, X_onehot, y, cv=3, scoring='r2')
    n_est_scores.append(scores.mean())

In [ ]:

max_depth_range = [None, 10, 20, 30, 40]
max_depth_scores = []

for max_depth in max_depth_range:
    et = ExtraTreesRegressor(n_estimators=200, max_depth=max_depth, random_state=42, n_jobs=-1)
    scores = cross_val_score(et, X_onehot, y, cv=3, scoring='r2')
    max_depth_scores.append(scores.mean())


In [ ]:
min_split_range = [2, 5, 10, 20]
min_split_scores = []

for min_split in min_split_range:
    et = ExtraTreesRegressor(n_estimators=200, min_samples_split=min_split, random_state=42, n_jobs=-1)
    scores = cross_val_score(et, X_onehot, y, cv=3, scoring='r2')
    min_split_scores.append(scores.mean())

In [ ]:
# Feature importance analysis with best model
feature_names = X_onehot.columns
best_et.fit(X_onehot, y)
feature_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': best_et.feature_importances_
}).sort_values('importance', ascending=False)

In [ ]:
# Random Forest comparison
from sklearn.ensemble import RandomForestRegressor
rf_optimized = RandomForestRegressor(
    n_estimators=grid_search.best_params_.get('n_estimators', 200),
    max_depth=grid_search.best_params_.get('max_depth', None),
    min_samples_split=grid_search.best_params_.get('min_samples_split', 2),
    min_samples_leaf=grid_search.best_params_.get('min_samples_leaf', 1),
    max_features=grid_search.best_params_.get('max_features', 'sqrt'),
    random_state=42,
    n_jobs=-1
)

rf_scores = cross_val_score(rf_optimized, X_onehot, y, cv=5, scoring='r2')

In [35]:

# Learning curve
from sklearn.model_selection import learning_curve

train_sizes, train_scores, val_scores = learning_curve(
    best_et, X_onehot, y,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=3, scoring='r2', n_jobs=-1
)

In [36]:
# Results
print(f"Baseline Extra Trees: {baseline_scores.mean():.4f}")
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best Extra Trees: {best_scores.mean():.4f}")
print(f"Improvement: {improvement:+.4f}")
print(f"Optimized Random Forest: {rf_scores.mean():.4f}")
print("Top 10 features:")
print(feature_importance.head(10))

Baseline Extra Trees: 0.6116
Best parameters: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 100}
Best Extra Trees: 0.5513
Improvement: -0.0603
Optimized Random Forest: 0.5738
Top 10 features:
                   feature  importance
10        price_volatility    0.125164
1                    baths    0.121524
11   price_reduction_total    0.099967
2                     sqft    0.095685
913       county_Nantucket    0.046973
829          zip_code_2554    0.046320
0                     beds    0.045068
238       region_Nantucket    0.040337
51           region_Boston    0.019564
8    parking_quality_score    0.019441
